### Solution at a glance
| | |
|---|---|
| Model | **LightGBM**, 5-fold Stratified CV ensemble (no AutoML — rulebook compliant) |
| Supporting features | Out-of-fold player target-encoding, within-game xG-share / "expected goals", team-goals, position, minutes |
| **CV Average Precision** | **0.6690** (random baseline = 0.085 → ~8× better) |

### How the score was built
| Stage | Idea added | CV AP |
|---|---|---|
| v1 | team_goals + player target-encoding + xG interactions | 0.426 |
| v2 | within-game xG-share / expected-goals | 0.428 |
| **v3** | **+ goal-accounting (`remaining_cap`)** | **0.6690** |


In [ ]:
import warnings, gc, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
import lightgbm as lgb

SEED, N_FOLDS = 42, 5

def find_path(fname):
    for base in [".", "/kaggle/input"]:
        if base == "/kaggle/input" and os.path.isdir(base):
            for d in os.listdir(base):
                p = os.path.join(base, d, fname)
                if os.path.exists(p): return p
        elif os.path.exists(os.path.join(base, fname)):
            return os.path.join(base, fname)
    return fname
TRAIN_PATH, TEST_PATH = find_path("train.csv"), find_path("test.csv")
print(TRAIN_PATH, TEST_PATH)

In [ ]:
train = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH)
print("train:", train.shape, "| test:", test.shape)
y = train["scored_flag"].astype(int).values
test_ids = test["appearance_id"].values
ntr = len(train)
train["_is_test"], test["_is_test"] = 0, 1
full = pd.concat([train, test], ignore_index=True, sort=False)
print("positive rate:", round(y.mean(), 4))

## Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
train["scored_flag"].astype(int).value_counts().plot.bar(ax=ax[0], color=["#4C72B0","#DD8452"])
ax[0].set_title(f"Target balance (pos rate={y.mean():.3f})")
train.groupby("position")["scored_flag"].mean().sort_values().plot.barh(ax=ax[1], color="#55A868")
ax[1].set_title("Scoring rate by position")
is_home = train["home_away"].astype(str).str.upper().eq("HOME")
tg = np.where(is_home, train["home_club_goals"], train["away_club_goals"])
pd.Series(train["scored_flag"].values).groupby(np.clip(tg,0,6)).mean().plot(ax=ax[2], marker="o", color="#C44E52")
ax[2].set_title("Scoring rate vs team goals"); ax[2].set_xlabel("team goals")
plt.tight_layout(); plt.show()

**Key findings**

1. **Imbalanced** (~8.5% positives) → AP is the right metric.
2. **Position**: attackers ~17%, midfielders ~8%, defenders ~4%, keepers ~0%.
3. **Team goals are a hard gate**: when a player's team scores 0 (~27% of rows) the scoring rate ≈ 0%. `home_club_goals`/`away_club_goals` are present in `test.csv`, so this is a legitimate, dominant feature.
4. **Random split**: train & test cover the *same* seasons (2012–2025) and share 99.8% of players & games → player history and *within-game* structure are valid, powerful signals.

## Feature engineering

### Basic + match context

In [ ]:
parts = full["appearance_id"].str.split("_", n=1, expand=True)
full["game_id"], full["player_id"] = parts[0], parts[1]
dt = pd.to_datetime(full["date"], errors="coerce")
full["month"], full["dayofweek"], full["year"] = dt.dt.month, dt.dt.dayofweek, dt.dt.year

is_home = full["home_away"].astype(str).str.upper().eq("HOME")
full["is_home"] = is_home.astype(int)
full["team_goals"] = np.where(is_home, full["home_club_goals"], full["away_club_goals"])
full["opp_goals"]  = np.where(is_home, full["away_club_goals"], full["home_club_goals"])
full["total_goals"] = full["home_club_goals"] + full["away_club_goals"]
full["team_scored_any"] = (full["team_goals"] > 0).astype(int)
full["goal_diff_signed"] = full["team_goals"] - full["opp_goals"]

### Goal-accounting the decisive feature

The same match appears in both train and test, and the number of goals a team scored
(`team_goals`) is fixed. Some of those goals are already **claimed** by *known* (train)
teammates whose `scored_flag = 1`. What's left is the **remaining capacity**:

$$\text{remaining\_cap} = \text{team\_goals} - \sum_{\text{train teammates}} \text{scored\_flag}$$

If `remaining_cap ≤ 0`, every goal is already accounted for → **this player almost
certainly did not score**. We compute it leave-one-out: a *train* row subtracts its own
label; a *test* row subtracts 0 (its label is unknown) — identical, leakage-safe logic for both.

> *On its own this single feature scores AP ≈ 0.23 (vs 0.17 for team_goals) and makes ~56% of rows near-certain zeros — that is the jump from 0.43 to 0.6690.*

In [ ]:
scored_train = full["scored_flag"].fillna(0.0).astype(float)   # label for train, 0 for test
gid, ih = full["game_id"], full["is_home"]
grp_scored_sum = scored_train.groupby([gid, ih]).transform("sum")
full["scored_by_mates"] = grp_scored_sum - scored_train        # leave-one-out
full["remaining_cap"]    = full["team_goals"] - full["scored_by_mates"]
full["remaining_cap_pos"] = full["remaining_cap"].clip(lower=0)
full["cap_is_zero"]       = (full["remaining_cap"] <= 0).astype(float)
full["frac_goals_claimed"] = full["scored_by_mates"] / (full["team_goals"] + 1e-6)
full["grp_train_count"] = (full["_is_test"] == 0).astype(float).groupby([gid, ih]).transform("sum")
full["grp_test_count"]  = (full["_is_test"] == 1).astype(float).groupby([gid, ih]).transform("sum")
full["cap_per_uncertain"] = full["remaining_cap_pos"] / (full["grp_test_count"] + 1.0)

chk = full[full["_is_test"] == 0].copy(); chk["t"] = y
print(chk.groupby(chk["remaining_cap"].clip(-1, 4))["t"].agg(["mean", "count"]))

### Within-game attacking share & expected goals
For each (game, team) we sum every teammate's xG / shots / key-passes, then take *this
player's share* — combined with the goals actually scored, this estimates the player's
expected goals in **this** match, and (with `remaining_cap`) who is likely to grab a *remaining* goal.

In [ ]:
grp = ["game_id", "is_home"]
full["lineup_size"] = full.groupby(grp)["player_id"].transform("count")
for c in ["avg_xG","avg_npxG","avg_shots","avg_key_passes","avg_xGChain","minutes_played"]:
    filled = full[c].fillna(0.0)
    full[c+"_teamsum"] = filled.groupby([gid, ih]).transform("sum")
    full[c+"_share"]   = filled / (full[c+"_teamsum"] + 1e-6)
    full[c+"_rank"]    = full.groupby(grp)[c].rank(ascending=False, method="min")
    full[c+"_isTop"]   = (full[c+"_rank"] == 1).astype(float)

full["exp_goals_xG"]    = full["avg_xG_share"]   * full["team_goals"]
full["exp_goals_npxG"]  = full["avg_npxG_share"] * full["team_goals"]
full["exp_goals_shots"] = full["avg_shots_share"]* full["team_goals"]
full["xGshare_x_starter"] = full["avg_xG_share"] * full["starter_flag"].astype(float)
full["xGshare_x_min"]     = full["avg_xG_share"] * full["minutes_ratio"]
# killer combos: of the goals still available, who is the likely scorer?
full["cap_x_xGshare"]   = full["remaining_cap_pos"] * full["avg_xG_share"]
full["cap_x_npxGshare"] = full["remaining_cap_pos"] * full["avg_npxG_share"]
full["cap_x_shotshare"] = full["remaining_cap_pos"] * full["avg_shots_share"]
full["cap_x_xGtop"]     = full["remaining_cap_pos"] * full["avg_xG_isTop"]
full["exprem_goals_xG"] = full["avg_xG_share"] * full["remaining_cap_pos"]

for c in ["avg_xG","avg_npxG","avg_shots","avg_key_passes"]:
    full[c+"_x_teamgoals"] = full[c] * full["team_goals"]
    full[c+"_x_min"]       = full[c] * full["minutes_ratio"]
full["xg_per_shot"]       = full["avg_xG"] / (full["avg_shots"] + 1e-3)
full["intl_goal_rate"]    = full["international_goals"] / (full["international_caps"] + 1)
full["minutes_x_starter"] = full["minutes_played"] * full["starter_flag"].astype(float)

bool_cols = [c for c in full.columns if full[c].dtype == bool
             or set(pd.Series(full[c]).dropna().unique()) <= {True, False}]
for c in bool_cols:
    if c != "scored_flag": full[c] = full[c].astype(float)
print("features after FE:", full.shape[1])

### Categorical encoding
Low-cardinality → integer codes (LightGBM native categoricals). High-cardinality
(player, clubs, referee, stadium…) → **out-of-fold smoothed target encoding** + frequency.

In [ ]:
cat_native = ["foot","position","sub_position","competition_type","confederation",
              "country_name","market_value_tier","age_bucket","home_away"]
for c in cat_native:
    codes = pd.Categorical(full[c].astype(str)).codes
    full[c] = np.where(codes < 0, codes.max()+1, codes)

train = full.iloc[:ntr].copy(); test = full.iloc[ntr:].copy()
del full; gc.collect()

te_cols = ["player_id","home_club_name","away_club_name","referee","stadium","country_of_citizenship"]
GLOBAL, SMOOTH = y.mean(), 20.0
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(train, y))

def smap(keys, labels):
    s = pd.DataFrame({"k": keys, "y": labels})
    a = s.groupby("k")["y"].agg(["mean","count"])
    return (a["mean"]*a["count"] + GLOBAL*SMOOTH) / (a["count"] + SMOOTH)

for col in te_cols:
    oof = np.full(len(train), GLOBAL, np.float32)
    trc, tec = train[col].astype(str).values, test[col].astype(str).values
    for tr_idx, va_idx in folds:                       # out-of-fold
        enc = smap(trc[tr_idx], y[tr_idx])
        oof[va_idx] = pd.Series(trc[va_idx]).map(enc).fillna(GLOBAL).values
    encf = smap(trc, y)
    train[col+"_te"] = oof
    test[col+"_te"]  = pd.Series(tec).map(encf).fillna(GLOBAL).values.astype(np.float32)
    vc = pd.Series(np.concatenate([trc, tec])).value_counts()
    train[col+"_freq"] = pd.Series(trc).map(vc).values
    test[col+"_freq"]  = pd.Series(tec).map(vc).values

DROP = set(["name_x","name_y","home_club_id","away_club_id","appearance_id",
            "scored_flag","date","game_id","_is_test"] + te_cols)
feature_cols = [c for c in train.columns if c not in DROP and c in test.columns and train[c].dtype != object]
cat_idx = [feature_cols.index(c) for c in cat_native if c in feature_cols]
print(len(feature_cols), "model features")

### Train — LightGBM 5-fold CV

In [ ]:
X, Xt = train[feature_cols], test[feature_cols]
params = dict(objective="binary", metric="average_precision", learning_rate=0.05,
              num_leaves=255, min_child_samples=100, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=2.0, n_jobs=-1, verbose=-1, seed=SEED)

oof_pred, test_pred, models = np.zeros(len(train)), np.zeros(len(test)), []
for fi, (tr_idx, va_idx) in enumerate(folds):
    dtr = lgb.Dataset(X.iloc[tr_idx], y[tr_idx], categorical_feature=cat_idx)
    dva = lgb.Dataset(X.iloc[va_idx], y[va_idx], categorical_feature=cat_idx)
    m = lgb.train(params, dtr, num_boost_round=2500, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    oof_pred[va_idx] = m.predict(X.iloc[va_idx], num_iteration=m.best_iteration)
    test_pred += m.predict(Xt, num_iteration=m.best_iteration) / N_FOLDS
    models.append(m)
    print(f"Fold {fi}  AP={average_precision_score(y[va_idx], oof_pred[va_idx]):.5f}  iter={m.best_iteration}")
print(f"\n===== OVERALL OOF AP = {average_precision_score(y, oof_pred):.5f} =====")

### Feature importance & football inference

In [ ]:
imp = pd.DataFrame({"feature": feature_cols, "gain": models[-1].feature_importance("gain")}
                  ).sort_values("gain", ascending=False).head(20)
plt.figure(figsize=(8,7)); plt.barh(imp["feature"][::-1], imp["gain"][::-1], color="#4C72B0")
plt.title("Top 20 features (gain)"); plt.tight_layout(); plt.show()
imp.head(15)

**Inference**

* **Goal accounting dominates** — `remaining_cap` and its interactions (`cap × xG-share`) decide *whether a goal was even available* for this player, and *who* the likely taker is.
* **Finishing pedigree** — player target-encoding (career scoring rate) ranks among the top features: elite finishers convert more in identical contexts.
* **Quality × opportunity** — `exp_goals` (xG-share × team goals), `minutes_played`, `starter_flag`, and `sub_position` capture chance volume and tactical role.

In [ ]:
sub = pd.DataFrame({"appearance_id": test_ids, "scored_flag": test_pred})
sub.to_csv("solution.csv", index=False)
print("solution.csv:", sub.shape, "| pred mean:", round(sub.scored_flag.mean(), 4),
      "| train pos rate:", round(y.mean(), 4))
sub.head()